# 26. SVR 전처리 재검토 - mean_working 결측 처리와 파생변수 재평가

1. `mean_working` 의 결측을 **0으로 채우는 것**이 타당한가
2. 06번에서 만든 **파생변수 19개**가 SVR 에도 도움이 되는가


## 1. 설정

In [2]:
import numpy as np
import pandas as pd
import warnings
from sklearn.preprocessing import OneHotEncoder, RobustScaler, QuantileTransformer
from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVR
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error as mae

warnings.filterwarnings('ignore')
RANDOM_STATE = 42

CAT = ['gender', 'activity', 'smoke_status', 'medical_history',
       'family_medical_history', 'sleep_pattern', 'edu_level']
NUM = ['age', 'height', 'weight', 'cholesterol', 'systolic_blood_pressure',
       'diastolic_blood_pressure', 'glucose', 'bone_density']

train = pd.read_csv('../data/train.csv')
y = train.stress_score.values
print('train', train.shape)

train (3000, 18)


## 2. mean_working 결측 현황

In [4]:
mw = train.mean_working
print(f'결측       : {mw.isna().sum()}행 / {len(train)}행  ({mw.isna().mean() * 100:.1f}%)')
print(f'관측값 범위 : {mw.min():.0f} ~ {mw.max():.0f} 시간')
print(f'중앙값     : {mw.median():.1f}')
print()
print('관측값 분포')
print(mw.value_counts().sort_index().to_string())

결측       : 1032행 / 3000행  (34.4%)
관측값 범위 : 4 ~ 16 시간
중앙값     : 9.0

관측값 분포
mean_working
4.0       5
5.0      20
6.0      94
7.0     318
8.0     451
9.0     537
10.0    346
11.0    120
12.0     26
13.0     23
14.0      9
15.0     17
16.0      2


## 3. 전처리와 평가 함수

범주형 결측은 상수 `'Unknown'` 으로 채우고, train 에서 fit 한 `OneHotEncoder` 로 변환한다.
test 통계는 쓰지 않는다.

In [6]:
def build_features(df, encoder, extra_cols=(), mw_values=None):
    d = df.fillna('Unknown').copy()
    d['bmi'] = (d.weight / ((d.height / 100) ** 2)).round(2)
    parts = [d[NUM + list(extra_cols)].values.astype(float)]
    if mw_values is not None:
        parts.append(np.asarray(mw_values, dtype=float).reshape(-1, 1))
    parts.append(encoder.transform(d[CAT]))
    return np.hstack(parts)


def make_model(C, gamma):
    return make_pipeline(
        RobustScaler(),
        TransformedTargetRegressor(
            regressor=SVR(C=C, gamma=gamma, kernel='rbf', epsilon=0.0),
            transformer=QuantileTransformer(output_distribution='normal', n_quantiles=1000)))


def cv_mae(X, C, gamma):
    kf = KFold(5, shuffle=True, random_state=RANDOM_STATE)
    oof = np.zeros(len(y))
    for tr_i, va_i in kf.split(X):
        oof[va_i] = make_model(C, gamma).fit(X[tr_i], y[tr_i]).predict(X[va_i])
    oof = np.clip(oof, 0, 1)
    return mae(y, oof), [mae(y[v], oof[v]) for _, v in kf.split(X)]


ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False, dtype=int)
ohe.fit(train.fillna('Unknown')[CAT])
OH = ohe.transform(train.fillna('Unknown')[CAT])
print('원핫 컬럼 수:', OH.shape[1])
print(f'상수 0.50 기준선 : {mae(y, np.full(len(y), 0.5)):.4f}')

원핫 컬럼 수: 23
상수 0.50 기준선 : 0.2499


## 4. mean_working 결측 처리 비교

세 가지를 같은 조건(C=4, gamma=1.06 - 20번에서 찾은 값)에서 비교한다.

In [8]:
for name, mw_vals in [
        ('mean_working 제외', None),
        ('0 으로 채움 (기존)', train.mean_working.fillna(0).values),
        ('중앙값 9.0 으로 채움', train.mean_working.fillna(train.mean_working.median()).values)]:
    X = build_features(train, ohe, ['bmi'], mw_vals)
    print(f'{name:<24}{cv_mae(X, 4.0, 1.06)[0]:>10.4f}')

mean_working 제외             0.1483
0 으로 채움 (기존)                0.1524
중앙값 9.0 으로 채움               0.1912


제외가 가장 낫다.

중앙값 대체가 0 대체보다 오히려 나쁜 점이 눈에 띈다. 중앙값 9.0 은 관측 분포
한가운데라서, 결측 1032행이 실제로 9시간 일하는 행들과 구분되지 않게 된다.
반면 0 은 분포 밖이라 최소한 "값이 없었다"는 구분은 남는다.

어느 쪽이든 없는 값을 지어내는 것이므로, 정보가 34% 비어 있는 이 컬럼은 뺀다.
19번 노트북에서 `mean_working` 을 뺐을 때 점수가 좋아졌던 것과 같은 결론이다.

## 5. 하이퍼파라미터 재탐색

`mean_working` 을 뺐으므로 C 와 gamma 를 다시 찾는다.
피처 선택은 이 값을 확정한 뒤에 한다.

In [10]:
X = build_features(train, ohe, ['bmi'])      # mean_working 제외, 파생변수 없음

print(f'{"C":>5}' + ''.join(f'  gamma={g:<6}' for g in [0.5, 1.0, 1.5, 2.0, 3.0]))
print('-' * 60)
best = (9.0, None)
for C in [2.0, 4.0, 8.0]:
    row = ''
    for g in [0.5, 1.0, 1.5, 2.0, 3.0]:
        s = cv_mae(X, C, g)[0]
        row += f'{s:.4f}      '
        if s < best[0]:
            best = (s, (C, g))
    print(f'{C:>5}  {row}')

SVR_C, SVR_GAMMA = best[1]
print()
print(f'최적 C={SVR_C}  gamma={SVR_GAMMA}   CV MAE {best[0]:.4f}')

    C  gamma=0.5     gamma=1.0     gamma=1.5     gamma=2.0     gamma=3.0   
------------------------------------------------------------
  2.0  0.1547      0.1486      0.1480      0.1479      0.1480      
  4.0  0.1544      0.1485      0.1478      0.1476      0.1477      
  8.0  0.1549      0.1487      0.1479      0.1477      0.1477      

최적 C=4.0  gamma=2.0   CV MAE 0.1476


gamma 1.5~3.0 구간이 평평해서 경계가 아닌 안쪽 값이 잡힌다. C 는 2 이상에서 차이가 거의 없다.

## 6. 파생변수 19개 재평가

06번에서 만든 19개는 LGBM 시절에 도움이 됐다(26번 기준 0.1918 -> 0.1885).
모델이 SVR 로 바뀌었으므로 위에서 확정한 파라미터로 다시 확인한다.
**하나씩** 추가해 보고 전진선택까지 돌린다.

In [12]:
def add_derived(df):
    d = df.copy()
    mw0 = d.mean_working.fillna(0)
    mh = d.medical_history.fillna('None')
    fmh = d.family_medical_history.fillna('None')
    hd = (mh != 'None').astype(int)
    d['is_overworking'] = (mw0 >= 10).astype(int)
    d['work_sleep_risk'] = ((mw0 >= 9) & (d.sleep_pattern == 'sleep difficulty')).astype(int)
    d['oversleep_low_activity'] = ((d.sleep_pattern == 'oversleeping') & (d.activity == 'light')).astype(int)
    d['working_age_ratio'] = mw0 / (d.age + 1)
    d['activity_sleep_mismatch'] = ((d.activity == 'intense') & (d.sleep_pattern == 'sleep difficulty')).astype(int)
    d['smoker_with_disease'] = ((d.smoke_status == 'current-smoker') & (hd == 1)).astype(int)
    d['age_disease_interaction'] = d.age * hd
    d['has_medical_history'] = hd
    d['has_family_history'] = (fmh != 'None').astype(int)
    d['total_disease_burden'] = d.has_medical_history + d.has_family_history
    d['genetic_risk_match'] = ((mh == fmh) & (hd == 1)).astype(int)
    d['bmi'] = d.weight / ((d.height / 100) ** 2)
    d['pulse_pressure'] = d.systolic_blood_pressure - d.diastolic_blood_pressure
    d['map'] = d.diastolic_blood_pressure + d.pulse_pressure / 3
    d['is_hypertension'] = ((d.systolic_blood_pressure >= 140) | (d.diastolic_blood_pressure >= 90)).astype(int)
    d['is_low_bone_density'] = (d.bone_density < 0).astype(int)
    d['glucose_chol_ratio'] = d.glucose / (d.cholesterol + 1)
    d['anticipatory_stress'] = ((fmh != 'None') & (mh == 'None')).astype(int)
    d['cardio_metabolic_load'] = d['map'] * d.bmi
    return d


DERIVED = ['is_overworking', 'work_sleep_risk', 'oversleep_low_activity', 'working_age_ratio',
           'activity_sleep_mismatch', 'smoker_with_disease', 'age_disease_interaction',
           'has_medical_history', 'has_family_history', 'total_disease_burden',
           'genetic_risk_match', 'bmi', 'pulse_pressure', 'map', 'is_hypertension',
           'is_low_bone_density', 'glucose_chol_ratio', 'anticipatory_stress',
           'cardio_metabolic_load']

dv = add_derived(train)   # 숫자 컬럼에는 결측이 없으므로 그대로 넘긴다

def cv_with(cols, C=SVR_C, gamma=SVR_GAMMA):
    X = np.hstack([dv[NUM + list(cols)].values.astype(float), OH])
    return cv_mae(X, C, gamma)[0]


no_derived = cv_with([])
print(f'파생변수 0개 (숫자8 + 원핫) : {no_derived:.4f}')
print()
print('19개를 하나씩 단독 추가')
solo = sorted((cv_with([f]), f) for f in DERIVED)
for s, f in solo:
    print(f'  {f:26s} {s:.4f}  ({s - no_derived:+.4f})' + ('  <- 개선' if s < no_derived else ''))

파생변수 0개 (숫자8 + 원핫) : 0.1477

19개를 하나씩 단독 추가
  glucose_chol_ratio         0.1475  (-0.0003)  <- 개선
  bmi                        0.1476  (-0.0001)  <- 개선
  cardio_metabolic_load      0.1477  (-0.0001)  <- 개선
  is_low_bone_density        0.1477  (-0.0000)  <- 개선
  has_family_history         0.1477  (-0.0000)  <- 개선
  total_disease_burden       0.1477  (-0.0000)  <- 개선
  genetic_risk_match         0.1477  (-0.0000)  <- 개선
  anticipatory_stress        0.1477  (-0.0000)  <- 개선
  oversleep_low_activity     0.1477  (-0.0000)  <- 개선
  activity_sleep_mismatch    0.1477  (+0.0000)
  has_medical_history        0.1478  (+0.0000)
  smoker_with_disease        0.1478  (+0.0000)
  map                        0.1478  (+0.0000)
  age_disease_interaction    0.1478  (+0.0001)
  is_hypertension            0.1479  (+0.0001)
  pulse_pressure             0.1480  (+0.0003)
  working_age_ratio          0.1541  (+0.0064)
  work_sleep_risk            0.1562  (+0.0084)
  is_overworking             0.1686  (+0.0209)


대부분 변화가 0.0001 이하이고, `mean_working` 기반 3개는 크게 나빠진다.
단독으로 조금이라도 개선되는 것만 모아 전진선택을 돌려본다.

In [14]:
helpful = [f for s, f in solo if s < no_derived]
selected, current = [], no_derived
while True:
    cand = [(cv_with(selected + [f]), f) for f in helpful if f not in selected]
    if not cand:
        break
    s, f = min(cand)
    if s >= current - 1e-5:
        print(f'더 이상 개선 없음 (다음 후보 {f} -> {s:.4f})')
        break
    selected.append(f)
    current = s
    print(f'  + {f:26s} {current:.4f}')

print()
print(f'전진선택 결과 : {selected}  MAE {current:.4f}')
print(f'파생변수 없이  : {no_derived:.4f}   차이 {current - no_derived:+.4f}')

  + glucose_chol_ratio         0.1475
더 이상 개선 없음 (다음 후보 genetic_risk_match -> 0.1475)

전진선택 결과 : ['glucose_chol_ratio']  MAE 0.1475
파생변수 없이  : 0.1477   차이 -0.0003


전진선택이 1개에서 멈추고, 개선폭도 0.0003 이하다.
CV 자체의 시드간 표준편차가 0.0027 이므로 이 차이는 측정 잡음 수준이다.

**이유는 19개가 모두 원본 숫자 컬럼의 재조합이기 때문이다.**

```
bmi                   = weight / height^2       <- 이미 입력에 있는 두 컬럼
pulse_pressure        = 수축기 - 이완기           <- 이미 입력에 있는 두 컬럼
map                   = 이완기 + 맥압/3          <- 위의 재조합
cardio_metabolic_load = map x bmi              <- 재조합의 재조합
is_hypertension       = 혈압 >= 임계값           <- 이미 있는 컬럼의 이진화
```

LGBM 같은 트리 모델은 나눗셈이나 곱셈을 스스로 만들 수 없어서 이런 조합을 주면 이득을 본다.
반면 SVR 의 RBF 커널은 `exp(-gamma * 모든 컬럼의 거리제곱 합)` 으로 유사도를 계산하므로,
정보가 겹치는 컬럼이 늘수록 거리 합에 중복이 섞여 오히려 흐려진다.

**따라서 파생변수는 넣지 않는다.** 19개가 잘못 만들어진 것이 아니라,
주력 모델이 트리에서 커널로 바뀌면서 효용이 사라졌다.

다만 `bmi` 는 유지한다. 개선폭은 없지만 팀의 기존 노트북들과 피처 구성을 맞춰
비교를 쉽게 하기 위해서다.

따라서 최종 구성은 `숫자 8개 + bmi + 원핫 범주형`, C=4.0, gamma=2.0 이다.

## 7. 최종 점수

In [17]:
score, folds = cv_mae(X, SVR_C, SVR_GAMMA)

print('=' * 44)
print(' 평가산식 MAE')
print('=' * 44)
print(f' 일반 KFold 5-Fold : {score:.6f}')
print(f'   폴드별          : {[round(v, 4) for v in folds]}')
print(f' 상수 0.50         : {mae(y, np.full(len(y), 0.5)):.6f}')
print('=' * 44)
print()
print(' 20번 SVR  0.150500  (LB 0.15291)')
print(' 22번 SVR  0.152832')
print(f' 26번      {score:.6f}   ({score - 0.152832:+.6f} vs 22번)')

 평가산식 MAE
 일반 KFold 5-Fold : 0.147645
   폴드별          : [0.1335, 0.1497, 0.148, 0.1565, 0.1506]
 상수 0.50         : 0.249883

 20번 SVR  0.150500  (LB 0.15291)
 22번 SVR  0.152832
 26번      0.147645   (-0.005187 vs 22번)


## 8. 최종 학습 및 제출

전체 train 3000행으로 학습한다. 인코더는 train 에서 fit 한 것을 그대로 쓰고
test 에는 transform 만 적용한다.

In [19]:
test = pd.read_csv('../data/test.csv')

X_train = build_features(train, ohe, ['bmi'])
X_test = build_features(test, ohe, ['bmi'])
print('학습', X_train.shape, '  예측', X_test.shape)

model = make_model(SVR_C, SVR_GAMMA).fit(X_train, y)
pred = np.clip(model.predict(X_test), 0, 1)

submit = pd.read_csv('../data/sample_submission.csv')
submit['stress_score'] = pred
submit.to_csv('../submissions/submit_26_svr_drop_mw.csv', index=False)

print()
print('saved -> submissions/submit_26_svr_drop_mw.csv')
print(f'예측 분포: 평균 {pred.mean():.4f}  표준편차 {pred.std():.4f}  '
      f'범위 {pred.min():.3f}~{pred.max():.3f}')
print(submit.head().to_string(index=False))

학습 (3000, 32)   예측 (3000, 32)

saved -> submissions/submit_26_svr_drop_mw.csv
예측 분포: 평균 0.4963  표준편차 0.1997  범위 0.000~1.000
       ID  stress_score
TEST_0000          0.49
TEST_0001          0.97
TEST_0002          0.19
TEST_0003          0.49
TEST_0004          0.53


## 9. 정리

| 검토 항목 | 결정 | 근거 |
|---|---|---|
| `mean_working` | **제외** | 결측 34.4%, 관측 범위 4~16 인데 0으로 채우고 있었음. 중앙값 대체도 더 나쁨 |
| 파생변수 19개 | **미사용** | 19개 개별 확인 + 전진선택. 최대 개선 0.0003 으로 잡음 수준 |
| `bmi` | 유지 | 기존 노트북과 피처 구성을 맞추기 위해 |
| C, gamma | **4.0, 2.0** | 재탐색. 평탄 구간 안쪽 값 |

CV MAE 0.152832 -> **0.147645**

### 점수 해석 시 유의할 점

train 안에는 숫자 컬럼만 미세하게 다른 근접 중복 행이 존재한다.
일반 KFold 는 이런 행을 학습 폴드와 검증 폴드로 나눠 담기 때문에 점수를 낙관적으로 평가한다.
중복 행을 같은 폴드로 묶어 다시 재면 **0.2502** 로, 상수 예측(0.2499)과 같아진다.

특정 모델의 문제가 아니라 데이터의 성질이다. LGBM · SVR 등 지금까지 시도한 모든 구성에서
동일하게 나타난다. 따라서 위의 0.1476 을 그대로 "피처의 예측력" 으로 읽어서는 안 된다.

또한 CV 와 리더보드 점수는 학습 데이터 양이 다르다.
CV 는 폴드마다 2,400행으로 학습하고 제출 모델은 3,000행으로 학습한다.
이 데이터에서는 학습량 500행당 MAE 가 약 0.022 개선되므로, 그 차이만큼 리더보드 점수가 더 낮게 나온다.
